# Scaleworm Detection Verification Lab

<span style="font-family: 'Courier New', monospace;">

*AI-generated draft (Claude, Anthropic) — for review. All parameters and figures are derived from version-controlled scripts and data.*

This notebook runs the YOLO "Mushroom Model" on CAMHD video frames and lets you verify each detection. Your job is to look at each cropped detection and decide: **is this a scale worm, or not?**

**Workflow:**
1. **Choose your date range** — pick which days of video to analyze
2. **Extract frames** — pull Scene 1 frames from each video (the Mushroom vent zoom)
3. **Run the detector** — YOLO finds candidate scale worms at low confidence (catches more, but includes false positives)
4. **Verify each detection** — you'll see each crop at multiple zoom levels and mark it as worm or not-worm
5. **Export** — verified true detections are packaged as a YOLO-format dataset for downstream training
6. **Analyze** — generate abundance charts and model metrics from the verified dataset
7. **Fixed-frame analysis** — extract one frame per video at a consistent camera position for the most defensible abundance counts

</span>

## 1. Setup

<span style="font-family: 'Courier New', monospace;">

Run this cell first every time you open the notebook. It imports all required libraries and sets up the directory structure and configuration parameters used throughout the workflow. Check the output to confirm the model file and video archive are found at their expected paths.

</span>

In [ ]:
import json
import re
import shutil
import subprocess
import zipfile
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, clear_output
from PIL import Image

# ── Configuration ───────────────────────────────────────────────────
# YOLO model path — put mushroom.pt path here
MODEL_PATH = Path("/home/jovyan/scaleworm-student-lab/mushroom.pt")

# Video archive root
VIDEO_ROOT = Path("/home/jovyan/ooi/san_data/RS03ASHS-PN03B-06-CAMHDA301/")

# Working directory for this session
WORK_DIR   = Path("./verification_session")
FRAMES_DIR = WORK_DIR / "frames"
CROPS_DIR  = WORK_DIR / "crops"
EXPORT_DIR = WORK_DIR / "export"

# Frame extraction parameters
SCENE1_START_SEC    = 305   # Scene 1 start offset in each video
SCENE1_DURATION_SEC = 15    # Scene 1 duration
FPS                 = 1     # Frames per second to extract
FRAME_W, FRAME_H    = 1920, 1080

# Detection parameters
CONF_THRESHOLD = 0.1    # Low threshold to catch more candidates
MAX_BOX_SIZE   = 300    # Filter out boxes larger than this (not real worms)

# Standard 3-hour cadence times (UTC)
STANDARD_TIMES = {
    "T001500", "T031500", "T061500", "T091500",
    "T121500", "T151500", "T181500", "T211500",
}

# Color palette used throughout all charts
TEAL   = '#2ec4b6'
ORANGE = '#e76f51'
GOLD   = '#e9c46a'

for d in [WORK_DIR, FRAMES_DIR, CROPS_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Model:          {MODEL_PATH}")
print(f"  Exists:       {MODEL_PATH.exists()}")
print(f"Video root:     {VIDEO_ROOT}")
print(f"  Exists:       {VIDEO_ROOT.exists()}")
print(f"Work dir:       {WORK_DIR.resolve()}")
print(f"Conf threshold: {CONF_THRESHOLD}")

## 2. Choose Your Date Range

<span style="font-family: 'Courier New', monospace;">

Set your start and end dates below, then run the cell. It will scan the video archive and list all standard-cadence CAMHD videos within that range. CAMHD records at 8 fixed times per day (every 3 hours), so a full month should return around 240 videos. If fewer are listed, some recordings may be missing from the archive.

</span>

In [ ]:
def find_videos(video_root, start_date, end_date):
    """Find standard-cadence CAMHD videos between two dates (inclusive)."""
    import datetime
    d_start = datetime.date.fromisoformat(start_date)
    d_end   = datetime.date.fromisoformat(end_date)

    videos = []
    for mp4 in sorted(video_root.rglob("CAMHDA301-*.mp4")):
        m = re.search(r"CAMHDA301-(\d{4})(\d{2})(\d{2})T(\d{6})", mp4.name)
        if not m:
            continue
        y, mo, d, time_str = m.group(1), m.group(2), m.group(3), m.group(4)
        file_date = datetime.date(int(y), int(mo), int(d))
        if file_date < d_start or file_date > d_end:
            continue
        if f"T{time_str}" not in STANDARD_TIMES:
            continue
        videos.append(mp4)
    return sorted(videos)


# ════════════════════════════════════════════════════════════════════
# ▼▼▼  SET YOUR DATE RANGE HERE  ▼▼▼
# ════════════════════════════════════════════════════════════════════

START_DATE = "2024-10-01"   # First day to analyze (YYYY-MM-DD)
END_DATE   = "2024-10-31"   # Last day to analyze  (YYYY-MM-DD)

# ════════════════════════════════════════════════════════════════════

videos = find_videos(VIDEO_ROOT, START_DATE, END_DATE)

print(f"Found {len(videos)} standard-cadence videos "
      f"between {START_DATE} and {END_DATE}:\n")
for v in videos:
    print(f"  {v.name}")

## 3. Extract Scene 1 Frames

<span style="font-family: 'Courier New', monospace;">

Each CAMHD video is ~25 minutes long and follows a scripted pan/tilt sequence. Scene 1 (305–320 seconds) is the window when the camera zooms in on the Mushroom vent chimney — the habitat where *Lepidonotopodium piscesae* is most visible. This cell uses ffmpeg to extract frames from that 15-second window at 1 frame per second, producing up to 15 frames per video. Frames that have already been extracted are skipped automatically, so this cell is safe to rerun.

**Expected output:** A progress list showing each video and its frame count. Total frames should be approximately `number of videos × 15`.

</span>

In [ ]:
def extract_scene1_frames(video_path, output_dir):
    """Extract Scene 1 frames from a CAMHD video using ffmpeg.
    Returns the number of frames extracted, or 0 on failure.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    existing = sorted(output_dir.glob("frame_*.png"))
    if len(existing) >= FPS * SCENE1_DURATION_SEC - 1:
        return len(existing)

    cmd = [
        "ffmpeg", "-y",
        "-ss", str(SCENE1_START_SEC),
        "-i", str(video_path),
        "-t", str(SCENE1_DURATION_SEC),
        "-vf", f"fps={FPS}",
        "-q:v", "2",
        str(output_dir / "frame_%04d.png"),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    if result.returncode != 0:
        print(f"  ERROR extracting {video_path.name}: {result.stderr[-200:]}")
        return 0
    return len(sorted(output_dir.glob("frame_*.png")))


total_frames    = 0
video_frame_dirs = {}

for i, vpath in enumerate(videos):
    vid_name  = vpath.stem
    frame_dir = FRAMES_DIR / vid_name
    video_frame_dirs[vid_name] = frame_dir
    n = extract_scene1_frames(vpath, frame_dir)
    total_frames += n
    print(f"  [{i+1}/{len(videos)}] {vid_name}: {n} frames")

print(f"\nTotal: {total_frames} frames from {len(videos)} videos")

## 4. Run the YOLO Detector with Object Tracking

<span style="font-family: 'Courier New', monospace;">

The Mushroom Model is run on every extracted frame using a low confidence threshold (0.1) to capture as many candidate detections as possible. False positives are expected and will be filtered out in the verification step.

This cell uses **ByteTrack** object tracking (`model.track()`) rather than plain detection (`model.predict()`). The key difference is that tracking assigns a consistent ID to the same worm across multiple frames within a single video. Without tracking, the same worm sitting in frame across 15 frames would be counted 15 times. With tracking, it is counted once per unique track ID — a much more accurate representation of abundance.

The tracker is reset between videos (`model.predictor = None`) so that track IDs do not bleed across separate recording sessions.

**Expected output:** A list of videos with frame and detection counts, followed by a total candidate detection count.

</span>

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
from collections import defaultdict

model = YOLO(str(MODEL_PATH))
print(f"Loaded model: {MODEL_PATH.name}")

all_detections = []

for vid_name, frame_dir in sorted(video_frame_dirs.items()):
    frames = sorted(frame_dir.glob("frame_*.png"))
    if not frames:
        continue

    results = model.track(
        source=[str(f) for f in frames],
        conf=CONF_THRESHOLD,
        tracker="bytetrack.yaml",
        persist=True,
        verbose=False,
        stream=True,
    )

    vid_det_count = 0
    for frame_path, result in zip(frames, results):
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            continue

        xyxy      = boxes.xyxy.cpu().numpy()
        confs     = boxes.conf.cpu().numpy()
        track_ids = (
            boxes.id.cpu().numpy().astype(int)
            if boxes.id is not None
            else [-1] * len(xyxy)
        )

        for det_idx, (box, conf, tid) in enumerate(zip(xyxy, confs, track_ids)):
            x1, y1, x2, y2 = box
            w, h = x2 - x1, y2 - y1
            if w > MAX_BOX_SIZE or h > MAX_BOX_SIZE:
                continue

            all_detections.append({
                "video":      vid_name,
                "frame_file": frame_path.name,
                "frame_path": str(frame_path),
                "det_idx":    det_idx,
                "track_id":   int(tid),
                "x1": float(x1), "y1": float(y1),
                "x2": float(x2), "y2": float(y2),
                "conf":       float(conf),
                "label":      None,
            })
            vid_det_count += 1

    # Reset tracker between videos to prevent ID bleed across sessions
    model.predictor = None
    print(f"  {vid_name}: {len(frames)} frames, {vid_det_count} detections")

print(f"\nTotal: {len(all_detections)} candidate detections to verify")

## 5. Crop Detections

<span style="font-family: 'Courier New', monospace;">

Each detection bounding box is cropped from its source frame and saved as a small image to disk. A padding of 20 pixels is added on each side so you can see the surrounding context when reviewing each crop in the next step. Crops are saved to the `crops/` directory and referenced by index.

**Expected output:** A progress counter updating every 500 crops, ending with "Done."

</span>

In [ ]:
PAD_PX       = 20
_frame_cache = {}

def load_frame(frame_path):
    """Load a frame image with caching to avoid repeated disk reads."""
    if frame_path not in _frame_cache:
        _frame_cache[frame_path] = np.array(Image.open(frame_path))
        if len(_frame_cache) > 50:
            oldest = next(iter(_frame_cache))
            del _frame_cache[oldest]
    return _frame_cache[frame_path]


def crop_detection(det, pad=PAD_PX):
    """Crop a detection from its frame with padding. Returns numpy array."""
    img  = load_frame(det["frame_path"])
    h, w = img.shape[:2]
    x1   = max(0, int(det["x1"]) - pad)
    y1   = max(0, int(det["y1"]) - pad)
    x2   = min(w, int(det["x2"]) + pad)
    y2   = min(h, int(det["y2"]) + pad)
    return img[y1:y2, x1:x2]


print("Cropping detections...")
for i, det in enumerate(all_detections):
    crop      = crop_detection(det)
    crop_path = CROPS_DIR / f"crop_{i:06d}.png"
    Image.fromarray(crop).save(crop_path)
    det["crop_path"] = str(crop_path)

    if (i + 1) % 500 == 0 or (i + 1) == len(all_detections):
        print(f"  {i+1}/{len(all_detections)} crops saved")

_frame_cache.clear()
print("Done.")

## 6. Verify Detections

<span style="font-family: 'Courier New', monospace;">

This is the core manual review step. For each candidate detection, you will see three panels:

- **Left panel**: The crop at its original pixel size (1×)
- **Center panel**: The crop scaled up 2× for more detail
- **Right panel**: The crop scaled up 4× for fine detail inspection

Click **✓ Scale Worm** if it is a real *L. piscesae*, or **✗ Not a Worm** if it is a false detection (tube worm, bacterial mat, rock edge, artifact, etc.). Click **Skip** if you are genuinely unsure.

Your progress is saved to disk after every decision, so you can stop and resume at any time without losing work. When you reopen the notebook and rerun cells 1–5, your saved labels will be automatically restored before this cell runs.

**Expected output:** An interactive widget with the image panels and decision buttons. When all detections are reviewed, a completion message will appear — proceed to Step 7.

</span>

In [ ]:
SAVE_PATH = WORK_DIR / "verification_progress.json"

if SAVE_PATH.exists():
    with open(SAVE_PATH) as f:
        saved = json.load(f)
    for i, label in saved.get("labels", {}).items():
        idx = int(i)
        if idx < len(all_detections):
            all_detections[idx]["label"] = label
    print(f"Resumed progress: {len(saved.get('labels', {}))} detections already labeled")
else:
    print("Starting fresh — no saved progress found.")


def save_progress():
    labels = {str(i): d["label"] for i, d in enumerate(all_detections) if d["label"] is not None}
    with open(SAVE_PATH, "w") as f:
        json.dump({"labels": labels}, f, indent=2)


current_idx    = [0]
output_images  = widgets.Output(layout=widgets.Layout(width="100%"))
output_info    = widgets.Output(layout=widgets.Layout(width="100%"))
output_progress= widgets.Output(layout=widgets.Layout(width="100%"))

for i, det in enumerate(all_detections):
    if det["label"] is None:
        current_idx[0] = i
        break

btn_worm     = widgets.Button(description="✓ Scale Worm",  button_style="success",
                               layout=widgets.Layout(width="180px", height="50px"),
                               style={"font_weight": "bold"})
btn_not_worm = widgets.Button(description="✗ Not a Worm", button_style="danger",
                               layout=widgets.Layout(width="180px", height="50px"),
                               style={"font_weight": "bold"})
btn_skip     = widgets.Button(description="⟳ Skip",        button_style="warning",
                               layout=widgets.Layout(width="120px", height="50px"))
btn_prev     = widgets.Button(description="◀ Previous",
                               layout=widgets.Layout(width="120px", height="50px"))


def show_detection(idx):
    det  = all_detections[idx]
    crop = np.array(Image.open(det["crop_path"]))
    with output_images:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        axes[0].imshow(crop);           axes[0].set_title("1× (original)",  fontsize=12, fontweight="bold"); axes[0].axis("off")
        crop_2x = np.array(Image.fromarray(crop).resize((crop.shape[1]*2, crop.shape[0]*2), Image.NEAREST))
        axes[1].imshow(crop_2x);        axes[1].set_title("2× zoom",         fontsize=12, fontweight="bold"); axes[1].axis("off")
        crop_4x = np.array(Image.fromarray(crop).resize((crop.shape[1]*4, crop.shape[0]*4), Image.NEAREST))
        axes[2].imshow(crop_4x);        axes[2].set_title("4× zoom",         fontsize=12, fontweight="bold"); axes[2].axis("off")
        fig.tight_layout(); plt.show()
    with output_info:
        clear_output(wait=True)
        label_color = {"scale_worm": "🟢", "not_worm": "🔴", "skip": "🟡"}.get(det["label"], "⚪")
        print(f"Detection {idx+1}/{len(all_detections)}  |  Conf: {det['conf']:.3f}  |  "
              f"Track ID: {det['track_id']}  |  Video: {det['video']}  |  "
              f"Frame: {det['frame_file']}  |  Status: {label_color} {det['label'] or 'unlabeled'}")
    update_progress()


def update_progress():
    n_done  = sum(1 for d in all_detections if d["label"] is not None)
    n_worm  = sum(1 for d in all_detections if d["label"] == "scale_worm")
    n_not   = sum(1 for d in all_detections if d["label"] == "not_worm")
    n_skip  = sum(1 for d in all_detections if d["label"] == "skip")
    n_total = len(all_detections)
    pct     = 100 * n_done / max(n_total, 1)
    with output_progress:
        clear_output(wait=True)
        bar = "█" * int(pct/2) + "░" * (50 - int(pct/2))
        print(f"Progress: [{bar}] {pct:.0f}%  ({n_done}/{n_total})  |  "
              f"🟢 {n_worm} worms  |  🔴 {n_not} not-worm  |  🟡 {n_skip} skipped")


def advance():
    start = current_idx[0]
    for offset in range(1, len(all_detections)+1):
        candidate = (start + offset) % len(all_detections)
        if all_detections[candidate]["label"] is None:
            current_idx[0] = candidate
            show_detection(current_idx[0])
            return
    current_idx[0] = len(all_detections) - 1
    show_detection(current_idx[0])
    with output_info:
        print("\n🎉  ALL DETECTIONS VERIFIED!  Proceed to Step 7 to review the summary.")


def on_worm(b):     all_detections[current_idx[0]]["label"] = "scale_worm"; save_progress(); advance()
def on_not_worm(b): all_detections[current_idx[0]]["label"] = "not_worm";   save_progress(); advance()
def on_skip(b):     all_detections[current_idx[0]]["label"] = "skip";        save_progress(); advance()
def on_prev(b):
    if current_idx[0] > 0: current_idx[0] -= 1
    show_detection(current_idx[0])

btn_worm.on_click(on_worm); btn_not_worm.on_click(on_not_worm)
btn_skip.on_click(on_skip); btn_prev.on_click(on_prev)

display(widgets.VBox([
    output_progress,
    output_images,
    output_info,
    widgets.HBox([btn_prev, btn_worm, btn_not_worm, btn_skip],
                 layout=widgets.Layout(justify_content="center", gap="10px")),
]))
show_detection(current_idx[0])

## 7. Verification Summary

<span style="font-family: 'Courier New', monospace;">

Run this cell at any time to see a breakdown of your verification progress. It shows how many detections have been confirmed as scale worms, rejected as false positives, skipped, or left unlabeled. The false positive rate tells you what proportion of the model's detections were incorrect — this is a key model performance metric.

You can also see per-day tracking statistics here, including how many unique track IDs were found versus total raw detections. The ratio of detections to unique tracks (`dets_per_track`) tells you on average how many frames each worm was visible for — a useful indicator of whether the tracker is working correctly.

**Do not export** (Step 8) until all detections are labeled and `Unlabeled: 0` appears.

</span>

In [ ]:
import pandas as pd

n_total    = len(all_detections)
n_worm     = sum(1 for d in all_detections if d["label"] == "scale_worm")
n_not      = sum(1 for d in all_detections if d["label"] == "not_worm")
n_skip     = sum(1 for d in all_detections if d["label"] == "skip")
n_unlabeled= sum(1 for d in all_detections if d["label"] is None)

print("=" * 60)
print("VERIFICATION SUMMARY")
print("=" * 60)
print(f"  Total detections:     {n_total:,}")
print(f"  ✓ Scale worm:         {n_worm:,} ({100*n_worm/max(n_total,1):.1f}%)")
print(f"  ✗ Not a worm:         {n_not:,} ({100*n_not/max(n_total,1):.1f}%)")
print(f"  ⟳ Skipped:            {n_skip:,}")
print(f"  ⚪ Unlabeled:          {n_unlabeled:,}")
print(f"  False positive rate:  {100*n_not/max(n_worm+n_not,1):.1f}%")
print("=" * 60)

if n_unlabeled > 0:
    print(f"\n⚠️  {n_unlabeled} detections still unlabeled. "
          "Return to Step 6 to finish before exporting.")

# ── Per-day tracking diagnostic ──────────────────────────────────────
print("\n" + "=" * 60)
print("TRACKING DIAGNOSTIC — confirmed scale worms only")
print("=" * 60)

track_df = pd.DataFrame([
    {
        'date':     d['video'][10:18],
        'track_id': d['track_id'],
        'conf':     d['conf'],
    }
    for d in all_detections if d['label'] == 'scale_worm'
])

if not track_df.empty:
    track_df['date'] = pd.to_datetime(track_df['date'], format='%Y%m%d')
    summary = track_df.groupby('date').agg(
        total_detections = ('track_id', 'count'),
        unique_tracks    = ('track_id', 'nunique'),
        mean_conf        = ('conf',     'mean')
    ).reset_index()
    summary['dets_per_track'] = summary['total_detections'] / summary['unique_tracks']
    print(summary.to_string(index=False))
    n_unique_tracks = track_df['track_id'].nunique()
    print(f"\n  Total unique track IDs (worms): {n_unique_tracks:,}")
    print(f"  Avg detections per track:       {n_worm / max(n_unique_tracks,1):.1f} frames")
else:
    print("  No confirmed scale worm detections yet.")

## 8. Export Verified Detections as YOLO Dataset

<span style="font-family: 'Courier New', monospace;">

This cell packages your verified true detections into a YOLO-format dataset ready for downstream model training. Only detections labeled **scale_worm** are included. Each source frame is copied into the dataset alongside a matching `.txt` label file.

**YOLO label format** — each line in a `.txt` file represents one detection:
```
class_id  center_x  center_y  width  height
```
All coordinates are normalized to [0, 1] relative to the 1920×1080 frame dimensions.

The dataset is also zipped for easy transfer. The resulting zip file can be uploaded to Ultralytics for further manual corrections, or used directly with the YOLO training command shown at the end of the output.

**Expected output:** A completion summary showing detection count, unique frame count, file paths, and zip size.

</span>

In [ ]:
def pixel_to_yolo(x1, y1, x2, y2, img_w=FRAME_W, img_h=FRAME_H):
    """Convert pixel coords to YOLO normalized (cx, cy, w, h), clamped to [0,1]."""
    cx = max(0.0, min(1.0, ((x1+x2)/2) / img_w))
    cy = max(0.0, min(1.0, ((y1+y2)/2) / img_h))
    w  = max(0.0, min(1.0, (x2-x1) / img_w))
    h  = max(0.0, min(1.0, (y2-y1) / img_h))
    return cx, cy, w, h


worm_dets = [d for d in all_detections if d["label"] == "scale_worm"]

if len(worm_dets) == 0:
    print("No verified worm detections to export!")
else:
    from collections import defaultdict
    frame_groups = defaultdict(list)
    for det in worm_dets:
        frame_groups[det["frame_path"]].append(det)

    img_dir = EXPORT_DIR / "images" / "train"
    lbl_dir = EXPORT_DIR / "labels" / "train"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for frame_path, dets in sorted(frame_groups.items()):
        unique_name = f"{dets[0]['video']}_{dets[0]['frame_file']}"
        stem        = Path(unique_name).stem
        dst_img     = img_dir / unique_name
        if not dst_img.exists():
            shutil.copy2(Path(frame_path), dst_img)
        with open(lbl_dir / f"{stem}.txt", "w") as f:
            for det in dets:
                cx, cy, w, h = pixel_to_yolo(det["x1"], det["y1"], det["x2"], det["y2"])
                f.write(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

    yaml_path = EXPORT_DIR / "dataset.yaml"
    with open(yaml_path, "w") as f:
        f.write(f"path: .\ntrain: images/train\nval: images/train\n\n")
        f.write(f"nc: 1\nnames:\n  0: scale_worm\n\n")
        f.write(f"# Verified detections: {len(worm_dets)}\n")
        f.write(f"# Source frames: {len(frame_groups)}\n")
        f.write(f"# Date range: {START_DATE} to {END_DATE}\n")
        f.write(f"# Confidence threshold: {CONF_THRESHOLD}\n")
        f.write(f"# Frame size: {FRAME_W}x{FRAME_H}\n")

    zip_name = f"verified_scaleworm_dataset_{START_DATE}_to_{END_DATE}.zip"
    zip_path = WORK_DIR / zip_name
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fpath in sorted(EXPORT_DIR.rglob("*")):
            if fpath.is_file():
                zf.write(fpath, fpath.relative_to(EXPORT_DIR))

    print("=" * 60)
    print("EXPORT COMPLETE")
    print("=" * 60)
    print(f"  Verified worm detections: {len(worm_dets):,}")
    print(f"  Unique source frames:     {len(frame_groups):,}")
    print(f"  YOLO dataset:             {EXPORT_DIR}")
    print(f"  Zip file:                 {zip_path}")
    print(f"  Zip size:                 {zip_path.stat().st_size/1e6:.1f} MB")
    print("=" * 60)
    print(f"\nTo train a new YOLO model on this dataset:")
    print(f"  yolo detect train data=dataset.yaml model=yolo11m.pt epochs=20 imgsz=1920")

## 9. Fixed-Frame Scene Position Viewer

<span style="font-family: 'Courier New', monospace;">

A key limitation of the multi-frame approach above is that the camera moves during the 15-second Scene 1 window, meaning different frames capture different parts of the vent. This makes cross-video comparisons inconsistent and can cause the same worm to appear at multiple positions across frames.

The recommended approach from the lab group is to extract **one frame per video at the same fixed offset**, ensuring every comparison uses the same camera field of view.

Use the dropdown and slider below to browse through October 2024 videos and find the offset (in seconds) where the camera is best positioned to see scale worms. Note the offset value shown under the image — you will use it in the next step.

**Expected output:** An interactive viewer showing a full-resolution frame. Scrub the slider between 305–320 seconds to find your ideal position, and switch videos using the dropdown to confirm the offset is consistent.

</span>

In [ ]:
SCENE1_START_SEC = 305
SCENE1_END_SEC   = 320
PREVIEW_DIR      = Path("./verification_session/position_preview")
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

# Filter to October 2024 videos only
videos_oct_2024 = sorted([
    v for v in VIDEO_ROOT.rglob("CAMHDA301-*.mp4")
    if "202410" in v.name
])
print(f"October 2024 videos available: {len(videos_oct_2024)}")

video_dropdown = widgets.Dropdown(
    options=[(v.name, v) for v in videos_oct_2024],
    description='Video:',
    layout=widgets.Layout(width='600px')
)
slider = widgets.IntSlider(
    value=SCENE1_START_SEC, min=SCENE1_START_SEC, max=SCENE1_END_SEC, step=1,
    description='Offset (s):', continuous_update=False,
    layout=widgets.Layout(width='600px')
)
offset_label = widgets.Label(value=f"Current offset: {slider.value}s into video")
output_view  = widgets.Output()


def extract_single_frame(video_path, offset_sec, out_path):
    cmd = ["ffmpeg", "-y", "-ss", str(offset_sec), "-i", str(video_path),
           "-frames:v", "1", "-q:v", "2", str(out_path)]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    return result.returncode == 0 and Path(out_path).exists()


def show_frame(video_path, offset):
    out_path = PREVIEW_DIR / f"{Path(video_path).stem}_offset_{offset:04d}.png"
    if not out_path.exists():
        if not extract_single_frame(video_path, offset, out_path):
            with output_view:
                clear_output(wait=True)
                print(f"ERROR: Could not extract frame at {offset}s from {Path(video_path).name}")
            return
    with output_view:
        clear_output(wait=True)
        img = np.array(Image.open(out_path))
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.imshow(img)
        ax.set_title(f"{Path(video_path).name}  |  Offset: {offset}s", color='white', fontsize=10)
        ax.axis('off')
        fig.patch.set_facecolor('#0d1117')
        plt.tight_layout(); plt.show()
        print(f"Video : {Path(video_path).name}")
        print(f"Offset: {offset}s  ← note this down if this position looks good")


def on_slider_change(change):
    offset_label.value = f"Current offset: {change['new']}s into video"
    show_frame(video_dropdown.value, change['new'])

def on_dropdown_change(change):
    show_frame(change['new'], slider.value)

slider.observe(on_slider_change, names='value')
video_dropdown.observe(on_dropdown_change, names='value')
show_frame(videos_oct_2024[0], slider.value)

display(widgets.VBox([
    widgets.Label("1. Pick a video from the dropdown"),
    video_dropdown,
    widgets.Label("2. Scrub the slider to find the clearest worm position"),
    slider, offset_label, output_view
]))

## 10. Extract One Fixed Frame Per Video

<span style="font-family: 'Courier New', monospace;">

Set `FIXED_OFFSET_SEC` to the offset you identified in Step 9, then run this cell. It extracts exactly one frame from every video in your date range at that fixed time offset, saving them to the `single_frames/` directory.

Because every frame is taken at the same moment in the camera's movement sequence, the field of view is consistent across all videos. This makes detection counts directly comparable between recordings — a worm count of 5 on October 3rd means the same thing as a count of 5 on October 28th.

Frames that already exist are skipped, so this cell is safe to rerun.

**Expected output:** A progress list showing each video with a ✓ or ✗, followed by total extracted and failed counts.

</span>

In [ ]:
import datetime

# ════════════════════════════════════════════════════════════════════
# ▼▼▼  SET YOUR CHOSEN OFFSET HERE (from Step 9)  ▼▼▼
# ════════════════════════════════════════════════════════════════════

FIXED_OFFSET_SEC = 320      # seconds into the video

# ════════════════════════════════════════════════════════════════════

SINGLE_FRAME_DIR = Path("./verification_session/single_frames")
SINGLE_FRAME_DIR.mkdir(parents=True, exist_ok=True)

d_start = datetime.date.fromisoformat(START_DATE)
d_end   = datetime.date.fromisoformat(END_DATE)

sf_videos = []
for mp4 in sorted(VIDEO_ROOT.rglob("CAMHDA301-*.mp4")):
    m = re.search(r"CAMHDA301-(\d{4})(\d{2})(\d{2})T(\d{6})", mp4.name)
    if not m:
        continue
    file_date = datetime.date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
    if file_date < d_start or file_date > d_end:
        continue
    if f"T{m.group(4)}" not in STANDARD_TIMES:
        continue
    sf_videos.append(mp4)

print(f"Found {len(sf_videos)} videos | Extracting frame at offset {FIXED_OFFSET_SEC}s\n")

extracted, failed = [], []
for i, vpath in enumerate(sorted(sf_videos)):
    out_name = f"{vpath.stem}_frame_fixed.png"
    out_path = SINGLE_FRAME_DIR / out_name
    if out_path.exists():
        print(f"  [{i+1}/{len(sf_videos)}] Already exists — skipping: {out_name}")
        extracted.append(out_path); continue
    cmd = ["ffmpeg", "-y", "-ss", str(FIXED_OFFSET_SEC), "-i", str(vpath),
           "-frames:v", "1", "-q:v", "2", str(out_path)]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    if result.returncode == 0 and out_path.exists():
        print(f"  [{i+1}/{len(sf_videos)}] ✓ {out_name}")
        extracted.append(out_path)
    else:
        print(f"  [{i+1}/{len(sf_videos)}] ✗ FAILED: {vpath.name}")
        failed.append(vpath)

print(f"\nExtracted : {len(extracted)} frames")
print(f"Failed    : {len(failed)} frames")
print(f"Output dir: {SINGLE_FRAME_DIR}")

## 11. Run YOLO on Fixed Frames

<span style="font-family: 'Courier New', monospace;">

The Mushroom Model is run on each of the single fixed frames extracted in Step 10. Because each frame is independent (there is no movement across frames to track), plain detection (`model.predict()`) is used here instead of tracking. The confidence threshold has been raised from 0.1 to 0.3 now that we are working with clean, consistent frames — this reduces false positives without needing a separate manual verification pass.

The detection count per frame is the most direct and defensible abundance estimate in this workflow: one frame, one consistent field of view, one count per video.

**Expected output:** A progress list showing each frame and its detection count, followed by the total across all frames and a preview of the results table.

</span>

In [ ]:
import pandas as pd
from ultralytics import YOLO

model          = YOLO(str(MODEL_PATH))
CONF_SF        = 0.3    # higher threshold for clean single frames
single_frame_detections = []
frames_to_run  = sorted(SINGLE_FRAME_DIR.glob("*.png"))

print(f"Running YOLO on {len(frames_to_run)} single frames (conf ≥ {CONF_SF})...\n")

for i, frame_path in enumerate(frames_to_run):
    m = re.search(r"CAMHDA301-(\d{8})T(\d{6})", frame_path.name)
    if not m:
        continue
    date_str, time_str = m.group(1), m.group(2)

    results   = model.predict(source=str(frame_path), conf=CONF_SF, verbose=False)
    boxes     = results[0].boxes
    det_count = 0

    if boxes is not None and len(boxes) > 0:
        for box, conf in zip(boxes.xyxy.cpu().numpy(), boxes.conf.cpu().numpy()):
            x1, y1, x2, y2 = box
            if (x2-x1) > MAX_BOX_SIZE or (y2-y1) > MAX_BOX_SIZE:
                continue
            single_frame_detections.append({
                'frame': frame_path.name,
                'date':  pd.to_datetime(date_str, format='%Y%m%d'),
                'time':  time_str,
                'conf':  float(conf),
                'x1': float(x1), 'y1': float(y1),
                'x2': float(x2), 'y2': float(y2),
            })
            det_count += 1

    print(f"  [{i+1}/{len(frames_to_run)}] {frame_path.name}  →  {det_count} detections")

sf_df = pd.DataFrame(single_frame_detections)
print(f"\nTotal detections across all single frames: {len(sf_df)}")
sf_df.head()

## 12. Fixed-Frame Abundance Charts

<span style="font-family: 'Courier New', monospace;">

These charts visualize scale worm abundance using the fixed-frame detection counts from Step 11. Four outputs are produced:

- **Per-video bar chart** — one bar per video, showing raw detection count and mean confidence. This is the highest-resolution view of abundance.
- **Daily abundance line chart** — detections summed by day with a 7-day rolling mean, showing the monthly trend.
- **Fixed frame vs original comparison** — overlays the fixed-frame counts against the original multi-frame detection counts to show how much the original approach was inflated by repeat detections of the same worm.
- **Distribution analysis** — histogram, Q-Q plot, box plot, and normality statistics to characterize the shape of the abundance data.

All charts are saved as PNG files to the `export/` directory.

</span>

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Aggregate — one row per video
per_video_sf = (
    sf_df.groupby(['date', 'time'])
    .agg(detections=('conf', 'count'), mean_conf=('conf', 'mean'))
    .reset_index().sort_values('date')
)

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

ax = axes[0]
ax.bar(range(len(per_video_sf)), per_video_sf['detections'],
       color=TEAL, edgecolor='#0d1117', linewidth=0.5, alpha=0.85)
ax.set_ylabel('Worm Detections\n(single fixed frame)', color=GOLD)
ax.set_title(
    r'$Lepidonotopodium\ piscesae$ — Fixed-Position Abundance'
    f'\nMushroom Vent, Axial Seamount  |  CAMHD OOI  |  Oct 2024  |  Frame offset: {FIXED_OFFSET_SEC}s',
    color=GOLD)
ax.tick_params(colors=GOLD); ax.grid(True, axis='y'); ax.set_facecolor('#161b22')

ax = axes[1]
ax.plot(range(len(per_video_sf)), per_video_sf['mean_conf'],
        color=GOLD, linewidth=1.8, marker='o', markersize=4)
ax.axhline(CONF_SF, color=ORANGE, linewidth=1, linestyle='--',
           label=f'Conf threshold ({CONF_SF})')
ax.set_ylabel('Mean Confidence', color=GOLD)
ax.set_xlabel('Video (chronological)', color=GOLD)
ax.set_ylim(0, 1); ax.tick_params(colors=GOLD)
ax.legend(facecolor='white', labelcolor='black', fontsize=9)
ax.grid(True); ax.set_facecolor('#161b22')

tick_positions = range(0, len(per_video_sf), 8)
tick_labels    = [per_video_sf.iloc[i]['date'].strftime('%b %d') for i in tick_positions]
for a in axes: a.set_xticks(list(tick_positions))
axes[1].set_xticklabels(tick_labels, rotation=35, ha='right', color=GOLD)
axes[0].set_xticklabels([], color=GOLD)

fig.patch.set_facecolor('#0d1117')
fig.tight_layout()
plt.savefig(EXPORT_DIR / 'fig_single_frame_abundance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_single_frame_abundance.png')

In [ ]:
# Daily abundance line chart
daily_sf = (
    per_video_sf.groupby('date')
    .agg(total_detections=('detections','sum'), mean_conf=('mean_conf','mean'),
         videos_that_day=('detections','count'))
    .reset_index().sort_values('date')
)

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(daily_sf['date'], daily_sf['total_detections'], alpha=0.22, color=TEAL)
ax.plot(daily_sf['date'], daily_sf['total_detections'],
        color=TEAL, linewidth=2, marker='o', markersize=5,
        label='Detections per day (single fixed frame per video)')
if len(daily_sf) >= 7:
    rolling = daily_sf.set_index('date')['total_detections'].rolling('7D').mean()
    ax.plot(rolling.index, rolling.values, color=GOLD, linewidth=2,
            linestyle='--', label='7-day rolling mean')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
plt.xticks(rotation=35, ha='right', color=GOLD)
ax.set_xlabel('Date', color=GOLD)
ax.set_ylabel(f'Worm Detections (fixed frame @ {FIXED_OFFSET_SEC}s)', color=GOLD)
ax.set_title(
    r'$Lepidonotopodium\ piscesae$ — Fixed-Position Abundance Over Time'
    '\nMushroom Vent, Axial Seamount  |  CAMHD OOI  |  Oct 2024', color=GOLD)
ax.tick_params(colors=GOLD)
ax.legend(facecolor='white', labelcolor='black')
ax.grid(True); ax.set_facecolor('#161b22'); fig.patch.set_facecolor('#0d1117')
fig.tight_layout()
plt.savefig(EXPORT_DIR / 'fig_fixed_frame_daily.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_fixed_frame_daily.png')

In [ ]:
# Rebuild original daily detections from exported label files for comparison
from datetime import datetime as dt_class

LABELS_DIR  = Path('./verification_session/export/labels/train')
FNAME_REGEX = re.compile(r'(\d{8})T(\d{6})_frame_(\d+)')
records     = []

for txt in sorted(LABELS_DIR.glob('*.txt')):
    m = FNAME_REGEX.search(txt.stem)
    if not m: continue
    try:
        dt = dt_class.strptime(f'{m.group(1)}{m.group(2)}', '%Y%m%d%H%M%S')
    except ValueError:
        continue
    with open(txt, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]
    for line in lines:
        if len(line.split()) >= 5:
            records.append({'date': dt.date(), 'filename': txt.stem})

orig_df = pd.DataFrame(records)
daily   = (orig_df.groupby('date').agg(detections=('filename','count'))
           .reset_index())
daily['date'] = pd.to_datetime(daily['date'])
print(f'Rebuilt: {len(daily)} days, {daily["detections"].sum()} total detections')

# Comparison chart
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(daily_sf['date'], daily_sf['total_detections'],
        color=TEAL, linewidth=2, marker='o', markersize=5,
        label='Fixed frame detections (320s)')
ax.fill_between(daily_sf['date'], daily_sf['total_detections'], alpha=0.15, color=TEAL)
ax.plot(daily['date'], daily['detections'],
        color=ORANGE, linewidth=2, marker='s', markersize=4,
        linestyle='--', label='Original raw detections (full Scene 1)', alpha=0.8)
ax.fill_between(daily['date'], daily['detections'], alpha=0.08, color=ORANGE)
merged = pd.merge(daily_sf, daily, on='date', suffixes=('_sf','_orig'), how='inner')
ax.fill_between(merged['date'], merged['total_detections'], merged['detections'],
                alpha=0.12, color=ORANGE, label='Reduction from fixed-frame approach')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
plt.xticks(rotation=35, ha='right', color=GOLD)
ax.set_xlabel('Date', color=GOLD); ax.set_ylabel('Detections', color=GOLD)
ax.set_title(
    r'$Lepidonotopodium\ piscesae$ — Fixed Frame vs Original Detection Counts'
    '\nMushroom Vent, Axial Seamount  |  CAMHD OOI  |  Oct 2024', color=GOLD)
ax.tick_params(colors=GOLD)
ax.legend(facecolor='white', labelcolor='black')
ax.grid(True); ax.set_facecolor('#161b22'); fig.patch.set_facecolor('#0d1117')
fig.tight_layout()
plt.savefig(EXPORT_DIR / 'fig_fixed_vs_original.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_fixed_vs_original.png')

In [ ]:
# Distribution analysis
import matplotlib.gridspec as gridspec
from scipy import stats

det_values        = daily_sf['total_detections'].values
stat_sw, p_value  = stats.shapiro(det_values)
is_normal         = p_value > 0.05
mu, sigma         = det_values.mean(), det_values.std()
skewness          = stats.skew(det_values)
kurtosis          = stats.kurtosis(det_values)

skew_interp    = ('Right-skewed (tail toward high counts)' if skewness > 0.5
                  else 'Left-skewed (tail toward low counts)' if skewness < -0.5
                  else 'Approximately symmetric')
norm_interp    = 'Likely normal (p > 0.05)' if is_normal else 'Not normal (p ≤ 0.05)'

fig = plt.figure(figsize=(15, 10))
fig.patch.set_facecolor('#0d1117')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# Histogram
ax1 = fig.add_subplot(gs[0, 0]); ax1.set_facecolor('#161b22')
ax1.hist(det_values, bins=12, color=TEAL, edgecolor='#0d1117', linewidth=0.8, density=True, alpha=0.85)
x = np.linspace(det_values.min(), det_values.max(), 200)
ax1.plot(x, stats.norm.pdf(x, mu, sigma), color=ORANGE, linewidth=2, linestyle='--', label='Normal curve')
ax1.axvline(mu, color=GOLD, linewidth=1.8, label=f'Mean: {mu:.1f}')
ax1.axvline(np.median(det_values), color='white', linewidth=1.8, linestyle=':', label=f'Median: {np.median(det_values):.1f}')
ax1.set_xlabel('Detections per Day', color=GOLD); ax1.set_ylabel('Density', color=GOLD)
ax1.set_title('Detection Distribution\nwith Normal Curve', color=GOLD)
ax1.tick_params(colors=GOLD); ax1.legend(facecolor='white', labelcolor='black', fontsize=9); ax1.grid(True, alpha=0.4)

# Q-Q plot
ax2 = fig.add_subplot(gs[0, 1]); ax2.set_facecolor('#161b22')
(osm, osr), (slope, intercept, r) = stats.probplot(det_values, dist='norm')
ax2.scatter(osm, osr, color=TEAL, s=30, zorder=3, alpha=0.85)
ax2.plot(osm, slope * np.array(osm) + intercept, color=ORANGE, linewidth=2, linestyle='--', label='Normal reference')
ax2.set_xlabel('Theoretical Quantiles', color=GOLD); ax2.set_ylabel('Sample Quantiles', color=GOLD)
ax2.set_title('Q-Q Plot\n(points on line = normal distribution)', color=GOLD)
ax2.tick_params(colors=GOLD); ax2.legend(facecolor='white', labelcolor='black', fontsize=9); ax2.grid(True, alpha=0.4)

# Box plot
ax3 = fig.add_subplot(gs[1, 0]); ax3.set_facecolor('#161b22')
ax3.boxplot(det_values, patch_artist=True, vert=True, widths=0.5,
            boxprops=dict(facecolor=TEAL, color=GOLD),
            medianprops=dict(color=ORANGE, linewidth=2.5),
            whiskerprops=dict(color=GOLD), capprops=dict(color=GOLD),
            flierprops=dict(marker='o', color=GOLD, markerfacecolor=ORANGE, markersize=6))
ax3.set_ylabel('Detections per Day', color=GOLD)
ax3.set_title('Box Plot\n(outliers shown as circles)', color=GOLD)
ax3.tick_params(colors=GOLD); ax3.set_xticklabels(['Oct 2024'], color=GOLD); ax3.grid(True, axis='y', alpha=0.4)

# Summary stats
ax4 = fig.add_subplot(gs[1, 1]); ax4.set_facecolor('#161b22'); ax4.axis('off')
stats_text = (
    f"  DISTRIBUTION SUMMARY\n  {'─'*30}\n"
    f"  Mean          : {mu:.2f}\n  Median        : {np.median(det_values):.2f}\n"
    f"  Std dev       : {sigma:.2f}\n  Min           : {det_values.min()}\n"
    f"  Max           : {det_values.max()}\n  {'─'*30}\n"
    f"  Skewness      : {skewness:.3f}\n  Kurtosis      : {kurtosis:.3f}\n"
    f"  {'─'*30}\n  Shapiro-Wilk\n  W stat        : {stat_sw:.3f}\n"
    f"  p-value       : {p_value:.4f}\n  {'─'*30}\n"
    f"  Shape         : {skew_interp}\n\n  Normality     : {norm_interp}\n"
)
ax4.text(0.05, 0.95, stats_text, transform=ax4.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace', color=GOLD,
         bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))
ax4.set_title('Summary Statistics', color=GOLD)

fig.suptitle(
    r'$Lepidonotopodium\ piscesae$ — Detection Distribution Analysis'
    '\nMushroom Vent, Axial Seamount  |  CAMHD OOI  |  Oct 2024',
    color=GOLD, fontsize=13)
plt.savefig(EXPORT_DIR / 'fig_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_distribution.png')